# SharePoint Permissions Audit

Enumerate and analyse permissions across a site collection:
site-collection admins, group membership, broken inheritance on webs and lists,
direct (non-group) user grants, and add-in principals with elevated rights.

> **Requires:** tenant admin or site collection admin credentials.  
> **Install:** `pip install Office365-REST-Python-Client[notebooks]`  
> **Docs:**  
> - [SharePoint REST permissions API](https://learn.microsoft.com/en-us/sharepoint/dev/sp-add-ins/determine-sharepoint-rest-service-endpoint-uris)  
> - [Understanding permission levels](https://learn.microsoft.com/en-us/sharepoint/understanding-permission-levels)

---


## 0 · Setup

In [1]:
from __future__ import annotations

import warnings
from datetime import datetime, timezone

import pandas as pd
from IPython.display import Markdown, display
from office365.sharepoint.client_context import ClientContext
from office365.sharepoint.principal.type import PrincipalType
from office365.sharepoint.sharing.role_type import RoleType
from tests import test_client_id, test_client_secret, test_team_site_url

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 500)

# ── Auth ───────────────────────────────────────────────────────────────────────
TARGET_SITE_URL = test_team_site_url  # site collection to audit

ctx = ClientContext(TARGET_SITE_URL).with_client_credentials(test_client_id, test_client_secret)

# ── Constants ─────────────────────────────────────────────────────────────────
ROLE_TYPE_LABELS = {
    RoleType.Administrator: "Full Control",
    RoleType.Contributor: "Contribute",
    RoleType.Editor: "Edit",
    RoleType.Reader: "Read",
    RoleType.Guest: "Limited Access",
    RoleType.WebDesigner: "Design",
    RoleType.Reviewer: "Approve",
    RoleType.RestrictedReader: "Restricted Read",
    RoleType.None_: "None",
    RoleType.System: "System",
}

PRINCIPAL_TYPE_LABELS = {
    PrincipalType.User: "User",
    PrincipalType.SharePointGroup: "SP Group",
    PrincipalType.SecurityGroup: "Security Group",
    PrincipalType.DistributionList: "Distribution List",
    PrincipalType.All: "All",
}


def _role_label(role_type_kind: int | None) -> str:
    return ROLE_TYPE_LABELS.get(role_type_kind, f"Custom ({role_type_kind})")


def _principal_label(principal_type: int | None) -> str:
    return PRINCIPAL_TYPE_LABELS.get(principal_type, str(principal_type))


def _flag(condition: bool) -> str:
    return "⚠️" if condition else "✅"


print(f"Target site: {TARGET_SITE_URL}")
print(f"Audit time:  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")

Target site: https://mediadev8.sharepoint.com/sites/team
Audit time:  2026-05-23 13:40 UTC


---
## 1 · Site Collection Administrators

Site collection admins have unconditional Full Control across the entire site
collection — they bypass all permission checks. This list should be short,
current, and reviewed regularly.


In [2]:
admins_result = ctx.site.get_site_administrators().execute_query()

rows = []
for admin in admins_result.value:
    rows.append(
        {
            "Name": admin.name,
            "Email": admin.email,
            "Login name": admin.loginName,
        }
    )

df_admins = pd.DataFrame(rows)
print(f"Site collection administrators: {len(df_admins)}")
display(df_admins)

Site collection administrators: 0


""


---
## 2 · Site Groups and Membership

All SharePoint groups defined at the site collection level, with each member's
principal type (user, security group, distribution list).
External / guest users are flagged — they appear as `is_share_by_email_guest_user`
or have a `#ext#` login suffix.


In [3]:
groups = ctx.web.site_groups.expand(["Users"]).get().execute_query()

rows = []
for group in groups:
    for user in group.users:
        is_guest = (
            getattr(user, "is_share_by_email_guest_user", False)
            or "#ext#" in (user.login_name or "").lower()
            or (user.email or "").lower().endswith("#ext#@")
        )
        rows.append(
            {
                "Group": group.title,
                "Member": user.title,
                "Login": user.login_name,
                "Email": user.email,
                "Principal type": _principal_label(user.principal_type),
                "Site admin": "✓" if user.is_site_admin else "",
                "Guest": "⚠️ Guest" if is_guest else "",
            }
        )

df_groups = pd.DataFrame(rows)
print(f"Groups: {len(groups)}   Members: {len(df_groups)}")
display(df_groups)

# Highlight groups with guest members
guests = df_groups[df_groups["Guest"] != ""]
if len(guests):
    display(Markdown(f"### ⚠️ {len(guests)} guest account(s) found in site groups"))
    display(guests[["Group", "Member", "Email", "Login"]])

Groups: 10   Members: 13


,Group,Member,Login,Email,Principal type,Site admin,Guest
0,Limited Access System Group For List 05e95a61-b013-48d5-8c8e-298fc20cc586,SLinkClaim.2ea8cba7-93d1-43a4-9344-c05655c1a3f2.6933ba41-53e9-4263-90da-32dc3aa4e925,c:0u.c|tenant|a.6933ba4153e9426390da32dc3aa4e925,,Security Group,,
1,Limited Access System Group For List 05e95a61-b013-48d5-8c8e-298fc20cc586,SLinkClaim.490b6987-fe9f-4972-b6ee-9ab917812aba.5599a712-1837-424c-b68d-be85fe4f8b16,c:0u.c|tenant|a.5599a7121837424cb68dbe85fe4f8b16,,Security Group,,
2,Limited Access System Group For List 05e95a61-b013-48d5-8c8e-298fc20cc586,SLinkClaim.89bb5e03-81ab-44c0-88b4-1f7e1bc67e3c.22255ee9-d986-46cf-b423-4ab9305f5194,c:0u.c|tenant|a.22255ee9d98646cfb4234ab9305f5194,,Security Group,,
3,Limited Access System Group For List 05e95a61-b013-48d5-8c8e-298fc20cc586,SLinkClaim.95216599-5194-4f11-ba35-04a8e5ccb198.9fe07d6d-2606-4c81-ad67-d898cb0d12ca,c:0u.c|tenant|a.9fe07d6d26064c81ad67d898cb0d12ca,,Security Group,,
4,Limited Access System Group For List 05e95a61-b013-48d5-8c8e-298fc20cc586,SLinkClaim.c80e5172-fd90-4518-80e7-ba902bd721e5.73edfc10-9a0c-4eb0-88e8-bed8273469cf,c:0u.c|tenant|a.73edfc109a0c4eb088e8bed8273469cf,,Security Group,,
5,Limited Access System Group For Web 2eab6cf3-7103-442c-a7d5-65e22119d17a,SLinkClaim.2ea8cba7-93d1-43a4-9344-c05655c1a3f2.6933ba41-53e9-4263-90da-32dc3aa4e925,c:0u.c|tenant|a.6933ba4153e9426390da32dc3aa4e925,,Security Group,,
6,Limited Access System Group For Web 2eab6cf3-7103-442c-a7d5-65e22119d17a,SLinkClaim.490b6987-fe9f-4972-b6ee-9ab917812aba.5599a712-1837-424c-b68d-be85fe4f8b16,c:0u.c|tenant|a.5599a7121837424cb68dbe85fe4f8b16,,Security Group,,
7,Limited Access System Group For Web 2eab6cf3-7103-442c-a7d5-65e22119d17a,SLinkClaim.89bb5e03-81ab-44c0-88b4-1f7e1bc67e3c.22255ee9-d986-46cf-b423-4ab9305f5194,c:0u.c|tenant|a.22255ee9d98646cfb4234ab9305f5194,,Security Group,,
8,Limited Access System Group For Web 2eab6cf3-7103-442c-a7d5-65e22119d17a,SLinkClaim.95216599-5194-4f11-ba35-04a8e5ccb198.9fe07d6d-2606-4c81-ad67-d898cb0d12ca,c:0u.c|tenant|a.9fe07d6d26064c81ad67d898cb0d12ca,,Security Group,,
9,Limited Access System Group For Web 2eab6cf3-7103-442c-a7d5-65e22119d17a,SLinkClaim.c80e5172-fd90-4518-80e7-ba902bd721e5.73edfc10-9a0c-4eb0-88e8-bed8273469cf,c:0u.c|tenant|a.73edfc109a0c4eb088e8bed8273469cf,,Security Group,,


---
## 3 · Role Definitions (Permission Levels)

Permission levels defined on this site collection. Custom levels beyond the
SharePoint defaults are worth noting — they often accumulate as one-off
workarounds and should be periodically reviewed.


In [4]:
role_defs = ctx.web.role_definitions.get().execute_query()

rows = []
for rd in role_defs:
    rows.append(
        {
            "ID": rd.id,
            "Name": rd.name,
            "Description": rd.description,
            "Role type": _role_label(rd.role_type_kind),
            "Custom": "⚠️" if rd.role_type_kind in (0, None) else "",
        }
    )

df_roles = pd.DataFrame(rows).sort_values("Role type")
print(f"Role definitions: {len(df_roles)}")
display(df_roles)

custom = df_roles[df_roles["Custom"] == "⚠️"]
if len(custom):
    display(Markdown(f"### ⚠️ {len(custom)} custom permission level(s) detected"))
    display(custom[["Name", "Description"]])

Role definitions: 9


,ID,Name,Description,Role type,Custom
5,1073741825,Limited Access,"Can view specific lists, document libraries, list items, folders, or documents when given permis...",Custom (1),
4,1073741826,Read,Can view pages and list items and download documents.,Custom (2),
7,1073741924,System.LimitedView,NaN,Custom (255),
8,1073741925,System.LimitedEdit,NaN,Custom (255),
3,1073741827,Contribute,"Can view, add, update, and delete list items and documents.",Custom (3),
1,1073741828,Design,"Can view, add, update, delete, approve, and customize.",Custom (4),
0,1073741829,Full Control,Has full control.,Custom (5),
2,1073741830,Edit,"Can add, edit and delete lists; can view, add, update and delete list items and documents.",Custom (6),
6,1073741833,Web-Only Limited Access,Can only view the web when given permissions.,Custom (9),


---
## 4 · Root Web — Direct Role Assignments

When `has_unique_role_assignments` is True the web has broken inheritance —
permissions are managed independently of the parent. Direct role assignments
on a web are a common source of over-permissioning.


In [5]:
web = (
    ctx.web.expand(["RoleAssignments/Member", "RoleAssignments/RoleDefinitionBindings"])
    .select(["HasUniqueRoleAssignments", "Title", "Url"])
    .get()
    .execute_query()
)

has_unique = web.has_unique_role_assignments
print(f"Web: {web.properties.get('Title')}  |  URL: {web.properties.get('Url')}")
inheritance_msg = "YES — unique permissions" if has_unique else "inherits from parent"
print(f"Broken inheritance: {_flag(not has_unique)} {inheritance_msg}")

rows = []
for ra in web.role_assignments:
    member = ra.member
    for rd in ra.role_definition_bindings:
        rows.append(
            {
                "Principal": member.properties.get("Title"),
                "Login": member.properties.get("LoginName"),
                "Principal type": _principal_label(member.properties.get("PrincipalType")),
                "Permission": rd.name,
                "Role type": _role_label(rd.role_type_kind),
            }
        )

df_web_ra = pd.DataFrame(rows)
print(f"\nDirect role assignments on root web: {len(df_web_ra)}")
display(df_web_ra)

Web: team  |  URL: https://mediadev8.sharepoint.com/sites/team
Broken inheritance: ✅ YES — unique permissions

Direct role assignments on root web: 18


,Principal,Login,Principal type,Permission,Role type
0,team Owners,team Owners,SP Group,Full Control,Custom (5)
1,team Owners,team Owners,SP Group,Web-Only Limited Access,Custom (9)
2,team Visitors,team Visitors,SP Group,Read,Custom (2)
3,team Visitors,team Visitors,SP Group,Web-Only Limited Access,Custom (9)
4,team Members,team Members,SP Group,Edit,Custom (6)
5,team Members,team Members,SP Group,Web-Only Limited Access,Custom (9)
6,SharingLinks.c80e5172-fd90-4518-80e7-ba902bd721e5.AnonymousView.73edfc10-9a0c-4eb0-88e8-bed82734...,SharingLinks.c80e5172-fd90-4518-80e7-ba902bd721e5.AnonymousView.73edfc10-9a0c-4eb0-88e8-bed82734...,SP Group,Web-Only Limited Access,Custom (9)
7,SLinkClaim.c80e5172-fd90-4518-80e7-ba902bd721e5.73edfc10-9a0c-4eb0-88e8-bed8273469cf,c:0u.c|tenant|a.73edfc109a0c4eb088e8bed8273469cf,Security Group,Web-Only Limited Access,Custom (9)
8,Limited Access System Group For List 05e95a61-b013-48d5-8c8e-298fc20cc586,Limited Access System Group For List 05e95a61-b013-48d5-8c8e-298fc20cc586,SP Group,Web-Only Limited Access,Custom (9)
9,Limited Access System Group For Web 2eab6cf3-7103-442c-a7d5-65e22119d17a,Limited Access System Group For Web 2eab6cf3-7103-442c-a7d5-65e22119d17a,SP Group,Web-Only Limited Access,Custom (9)


---
## 5 · Sub-webs with Broken Inheritance

Sub-sites that manage their own permissions independently of the root web.
Each is a potential island of unreviewed access — especially relevant for
legacy team sites created before modern SharePoint.


In [7]:
sub_webs = ctx.web.webs.select(["Title", "Url", "HasUniqueRoleAssignments"]).get().execute_query()

rows = []
for w in sub_webs:
    rows.append(
        {
            "Title": w.properties.get("Title"),
            "URL": w.properties.get("Url"),
            "Broken inheritance": "⚠️ YES" if w.has_unique_role_assignments else "✅ Inherits",
        }
    )

df_webs = pd.DataFrame(rows)
broken_webs = df_webs[df_webs["Broken inheritance"] == "⚠️ YES"]
print(f"Sub-webs: {len(df_webs)}   Broken inheritance: {len(broken_webs)}")
display(df_webs)

KeyError: 'Broken inheritance'

---
## 6 · Lists and Libraries with Broken Inheritance

Lists or libraries with unique permissions are the most common over-permission
pattern in SharePoint. Each one is a separate permission boundary that has to
be audited independently.


In [8]:
lists = (
    ctx.web.lists.select(["Title", "DefaultViewUrl", "HasUniqueRoleAssignments", "Hidden", "BaseType"])
    .filter("Hidden eq false")
    .get()
    .execute_query()
)

rows = []
for lst in lists:
    rows.append(
        {
            "Title": lst.properties.get("Title"),
            "URL": lst.properties.get("DefaultViewUrl"),
            "Broken inheritance": "⚠️ YES" if lst.has_unique_role_assignments else "✅ Inherits",
        }
    )

df_lists = pd.DataFrame(rows)
broken_lists = df_lists[df_lists["Broken inheritance"] == "⚠️ YES"]
print(f"Lists/Libraries: {len(df_lists)}   Broken inheritance: {len(broken_lists)}")
display(df_lists)

if len(broken_lists):
    display(Markdown("### ⚠️ Lists with unique permissions — review required"))
    display(broken_lists)

Lists/Libraries: 6   Broken inheritance: 1


,Title,URL,Broken inheritance
0,Documents,/sites/team/Shared Documents/Forms/AllItems.aspx,⚠️ YES
1,Events,/sites/team/Events/calendar.aspx,✅ Inherits
2,Form Templates,/sites/team/FormServerTemplates/Forms/All Forms.aspx,✅ Inherits
3,Site Assets,/sites/team/SiteAssets/Forms/AllItems.aspx,✅ Inherits
4,Site Pages,/sites/team/SitePages/Forms/ByAuthor.aspx,✅ Inherits
5,Style Library,/sites/team/Style Library/Forms/AllItems.aspx,✅ Inherits


### ⚠️ Lists with unique permissions — review required

,Title,URL,Broken inheritance
0,Documents,/sites/team/Shared Documents/Forms/AllItems.aspx,⚠️ YES


---
## 7 · Direct User Grants (non-group assignments)

Users assigned directly to a securable object rather than via a group.
Direct grants are hard to maintain, easy to forget, and create orphaned
access when the user leaves the organisation. All direct grants should
be migrated to group membership.


In [9]:
# Re-use root web role assignments fetched in section 4
direct_users = df_web_ra[df_web_ra["Principal type"] == "User"].copy()

print(f"Direct user grants on root web: {len(direct_users)}")

if len(direct_users):
    display(Markdown("### ⚠️ Direct user assignments detected — migrate to groups"))
    display(direct_users[["Principal", "Login", "Permission", "Role type"]])
else:
    print("✅ No direct user grants on root web.")

Direct user grants on root web: 0
✅ No direct user grants on root web.


---
## 8 · Add-in and ACS Principals

Legacy SharePoint Add-ins (ACS-based app principals) that have been granted
permissions. These bypass modern Entra ID conditional access policies and
should be inventoried and migrated to Entra app registrations where possible.

> **Note:** requires `FullControl` at the site collection level to enumerate.


In [12]:
addin_result = ctx.web.get_addin_principals_having_permissions_in_sites(
    server_relative_urls=[ctx.web.properties.get("ServerRelativeUrl", "/")]
).execute_query()

rows = []
for principal in addin_result.value.addinPrincipals or []:
    rows.append(
        {
            "App ID": principal.appIdentifier,
            "Display name": principal.title,
            "Site URL": principal.serverRelativeUrl,
            # "Permissions":  principal.permissionsGrant,
        }
    )

df_addins = pd.DataFrame(rows)
print(f"Add-in / ACS principals: {len(df_addins)}")
if len(df_addins):
    display(Markdown("### ⚠️ Legacy add-in principals found — review for migration to Entra app registrations"))
    display(df_addins)
else:
    print("✅ No legacy add-in principals found.")

ClientRequestException: ('-2147024891, System.UnauthorizedAccessException', 'Attempted to perform an unauthorized operation.', '403 Client Error: Forbidden for url: https://mediadev8.sharepoint.com/sites/team/_api/Web/GetAddinPrincipalsHavingPermissionsInSites')

---
## 9 · Audit Summary

Consolidated findings. Use this as a baseline — run monthly and compare
DataFrames to detect permission drift.


In [11]:
findings = [
    ("Site collection admins", len(df_admins)),
    ("Group members (total)", len(df_groups)),
    ("Guest accounts in groups", len(df_groups[df_groups["Guest"] != ""])),
    ("Custom permission levels", len(df_roles[df_roles["Custom"] == "⚠️"])),
    ("Root web — direct role assignments", len(df_web_ra)),
    ("Root web — direct user grants", len(direct_users)),
    ("Sub-webs with broken inheritance", len(broken_webs)),
    ("Lists/libraries with broken inheritance", len(broken_lists)),
    ("Legacy add-in / ACS principals", len(df_addins)),
]

df_summary = pd.DataFrame(findings, columns=["Finding", "Count"])
df_summary["Status"] = df_summary.apply(
    lambda r: "⚠️"
    if (
        (r["Finding"] == "Site collection admins" and r["Count"] > 5)
        or (
            r["Finding"] != "Site collection admins"
            and r["Count"] > 0
            and r["Finding"] not in ("Group members (total)", "Root web — direct role assignments")
        )
    )
    else "✅",
    axis=1,
)

display(df_summary.style.hide(axis="index"))
print(f"\nAudit completed: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")

NameError: name 'broken_webs' is not defined